# Hawkes Processes: Simulation, Inference, and Declustering

This project studies Hawkes processes and parameter inference using the Expectation–Maximization (EM) algorithm. 

We investigate:
* Self-exciting cascade structures
* Resolution of component collapse via Branching EM (Declustering)
* Parameter recovery accuracy and EM convergence behavior
* Asymptotic consistency with varying dataset sizes

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import time

from src.simulator import simulate_hawkes, generate_mixture_data
from src.inference import branching_em, branching_em_with_diagnostics, bimodal_branching_em

rng = np.random.default_rng(42)
plt.style.use('default')

## 1. Hawkes Process Background

A Hawkes process is a point process with conditional intensity:
$$\lambda(t) = \mu + \alpha \sum_{t_i < t} e^{-\beta(t - t_i)}$$

* $\mu$: Background intensity
* $\alpha$: Excitation strength
* $\beta$: Exponential decay rate
* $\eta = \alpha / \beta$: Branching ratio

In [ ]:
T = 500
events = simulate_hawkes(1.0, 0.6, 1.5, T, rng)

# 1. Draw Event Timeline
plt.figure(figsize=(10, 2))
plt.eventplot(events, color='black', linewidths=0.8)
plt.title("Hawkes Process: Event Timeline")
plt.xlabel("Time")
plt.yticks([])
plt.tight_layout()
plt.show()

# 2. Draw Conditional Intensity
def intensity_curve(events, mu, alpha, beta, tgrid):
    lam = np.zeros_like(tgrid)
    for i, t in enumerate(tgrid):
        past = events[events < t]
        lam[i] = mu + alpha * np.sum(np.exp(-beta * (t - past)))
    return lam

t_subset = 50
tgrid = np.linspace(0, t_subset, 1000)
lam = intensity_curve(events, 1.0, 0.6, 1.5, tgrid)

plt.figure(figsize=(10, 3))
plt.plot(tgrid, lam, color='red', linewidth=1.5)
events_subset = events[events < t_subset]
plt.vlines(events_subset, ymin=1.0, ymax=lam[np.searchsorted(tgrid, events_subset)-1], 
           color='black', alpha=0.3, linestyles='--')
plt.title(f"Conditional Intensity Function $\lambda(t)$ (First {t_subset} time units)")
plt.xlabel("Time")
plt.ylabel("Intensity")
plt.tight_layout()
plt.show()

# 3. Draw Cascade Analysis 
dt = np.diff(events)
plt.figure(figsize=(8, 4))
plt.hist(dt, bins=50, density=True, alpha=0.7, color='steelblue', 
         edgecolor='black', label=r'Hawkes Empirical $\Delta t$')

mean_dt = np.mean(dt)
x_fit = np.linspace(0, np.max(dt), 200)
y_fit = (1 / mean_dt) * np.exp(-x_fit / mean_dt)
plt.plot(x_fit, y_fit, color='red', linestyle='--', linewidth=2, label='Poisson Reference')

plt.title("Inter-event Time Distribution (Cascade Structure)")
plt.xlabel(r"Time between consecutive events ($\Delta t$)")
plt.ylabel("Density")
plt.xlim(0, np.percentile(dt, 95))
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 2. Inference via Branching EM Algorithm

Standard mixture models on a single time-line fail due to parameter unidentifiability and shared-history collinearity, leading to component collapse. We resolve this by employing a Branching Structure EM algorithm (Declustering).

**E-step:**
Compute probabilities of event $i$ being a background event ($p_{ii}$) or triggered by event $j$ ($p_{ij}$):
$$p_{ii} = \frac{\mu}{\lambda(t_i)}, \quad p_{ij} = \frac{\alpha e^{-\beta(t_i - t_j)}}{\lambda(t_i)}$$

**M-step:**
Maximize the complete-data log-likelihood to update $\mu, \alpha, \beta$.

In [ ]:
est_mu, est_alpha, est_beta, ll_history, P = branching_em(events, T)

bg_probs = np.diag(P)
is_background = bg_probs > 0.5

fig, axes = plt.subplots(2, 1, figsize=(10, 6))

axes[0].plot(ll_history, marker='o', color='black', markersize=4)
axes[0].set_title("EM Convergence (Log-Likelihood)")
axes[0].set_ylabel("Log-Likelihood")
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].scatter(events[is_background], np.zeros(np.sum(is_background)), 
                color='blue', alpha=0.7, label='Background Events ($\mu$)')
axes[1].scatter(events[~is_background], np.ones(np.sum(~is_background)), 
                color='red', alpha=0.7, label='Triggered Events (Cascades)')
axes[1].set_title("Events Clustered by Branching Responsibility")
axes[1].set_xlabel("Time")
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(['Background', 'Triggered'])
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Convergence Analysis (EM Diagnostics)

To verify the theoretical properties of the EM algorithm, we track the complete parameter trajectory. We demonstrate two key theoretical guarantees:
1. The monotonic increase of the log-likelihood function.
2. The linear convergence rate of the parameter sequence $\theta^{(n)}$, evidenced by the logarithmic decay of the step size $|\theta^{(n+1)} - \theta^{(n)}|$.

In [ ]:
# ── Parameter Inference and Convergence Diagnostics ────────────────────
import numpy as np
import matplotlib.pyplot as plt

print("Executing inference with analytical convergence tracking...")
est_mu, est_alpha, est_beta, ll_h, p_h, p_diffs, P_matrix = branching_em_with_diagnostics(events, T)

# Visualization of EM theoretical properties
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Monotonicity of the EM Algorithm
ax1.plot(ll_h, color='#1f77b4', linewidth=2, marker='o', markersize=4)
ax1.set_title("A. Log-Likelihood Ascent", fontsize=12, fontweight='bold')
ax1.set_xlabel("Iteration Step")
ax1.set_ylabel(r"Log-Likelihood $\mathcal{L}(\theta)$")
ax1.grid(True, alpha=0.3)

# Panel B: Linear Convergence of the Parameter Sequence
ax2.plot(range(1, len(p_diffs) + 1), p_diffs, color='#d62728', linewidth=2, marker='s', markersize=4)
ax2.set_yscale('log')
ax2.set_title("B. Parameter Convergence Rate", fontsize=12, fontweight='bold')
ax2.set_xlabel("Iteration Step")
ax2.set_ylabel(r"$||\theta_{n+1} - \theta_n||$ (log scale)")
ax2.grid(True, which="both", alpha=0.3)

# Panel C: Stochastic Declustering (Branching Matrix Heatmap)
# Restricting to the first 50 events for visual clarity of the triggering cascade
im = ax3.imshow(P_matrix[:50, :50], cmap='viridis', interpolation='nearest')
ax3.set_title("C. Branching Matrix (First 50 Events)", fontsize=12, fontweight='bold')
ax3.set_xlabel("Triggering Ancestor Event ($j$)")
ax3.set_ylabel("Target Offspring Event ($i$)")
plt.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 4. Mixture Model: Problem and Component Collapse

In this section, we construct a true mixture dataset by superimposing two independent Hawkes processes with distinct parameter sets (a slow background process and a fast self-exciting process). 

We first demonstrate the failure of the naive Gaussian-Mixture-style EM algorithm $p(t_i) = \pi_1 \lambda_1(t_i) + \pi_2 \lambda_2(t_i)$. Due to the shared historical event pool $\mathcal{H}_t$, the recursive excitation intensities exhibit strong multicollinearity, causing the mixture weights $\pi$ to inevitably collapse to $\{0, 1\}$.

In [ ]:
# ── 1. Synthetic Mixture Sequence Generation (K=2) ───────────────────
T_mix = 500

# Component 0: Slow, background-dominated process
events_1 = simulate_hawkes(mu=0.3, alpha=0.2, beta=1.0, T=T_mix, rng=rng)
labels_1 = np.zeros(len(events_1), dtype=int)

# Component 1: Fast, highly self-exciting process
events_2 = simulate_hawkes(mu=1.5, alpha=0.6, beta=2.0, T=T_mix, rng=rng)
labels_2 = np.ones(len(events_2), dtype=int)

# Merge and temporally sort the independent streams
all_events = np.concatenate([events_1, events_2])
all_labels = np.concatenate([labels_1, labels_2])
order = np.argsort(all_events)

events_mix = all_events[order]
labels_true = all_labels[order]

print(f"Total events in mixture (N) : {len(events_mix)}")
print(f"  ├─ Component 0 counts     : {len(events_1)}")
print(f"  └─ Component 1 counts     : {len(events_2)}")

# ── 2. Component Collapse in Naive Mixture EM ────────────────────────
def naive_mixture_em_collapse(events, max_iter=15):
    """
    Demonstrates that a naive shared-history mixture formulation 
    causes mixing weights to collapse entirely into one component,
    even when initialized near the ground-truth parameters.
    """
    n = len(events)
    dt = np.diff(events)
    pi = np.array([0.5, 0.5])
    mu, alpha, beta = [0.3, 1.5], [0.2, 0.6], [1.0, 2.0]
    pi_history = [pi.copy()]
    
    for it in range(max_iter):
        L = np.zeros((n, 2))
        for k in range(2):
            A = np.zeros(n)
            for i in range(1, n):
                A[i] = np.exp(-beta[k]*dt[i-1]) * (1 + A[i-1])
            L[:, k] = mu[k] + alpha[k] * A
            
        weighted = L * pi
        R = weighted / np.maximum(weighted.sum(axis=1, keepdims=True), 1e-12)
        pi = R.mean(axis=0)
        pi_history.append(pi.copy())
        
    return np.array(pi_history)

pi_hist = naive_mixture_em_collapse(events_mix)

# ── 3. Visualization of Weight Collapse ──────────────────────────────
plt.figure(figsize=(8, 4))
plt.plot(pi_hist[:, 0], marker='o', label=r'Component 0 ($\pi_0$)')
plt.plot(pi_hist[:, 1], marker='s', label=r'Component 1 ($\pi_1$)')

true_ratio_0 = len(events_1) / len(events_mix)
true_ratio_1 = len(events_2) / len(events_mix)

plt.axhline(true_ratio_0, color='blue', linestyle='--', alpha=0.3, label='True Ratio 0')
plt.axhline(true_ratio_1, color='orange', linestyle='--', alpha=0.3, label='True Ratio 1')

plt.title("Degeneracy of Mixing Weights in Naive EM", fontsize=12, fontweight='bold')
plt.xlabel("EM Iteration")
plt.ylabel(r"Estimated Mixing Weight ($\pi$)")
plt.legend()
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 5. Mixture Model: Resolution and Recovery

To resolve the unidentifiability, we must shift the mixture assumption from the macroscopic sequence level to the microscopic event-triggering level. 

By superimposing the processes, the generative model mathematically equates to a single Hawkes process with a **Bimodal Kernel**. We extend the Branching EM to simultaneously estimate $K=2$ sets of parameters $(\mu_k, \alpha_k, \beta_k)$. The responsibility matrices separate not only background from offspring, but also partition them into their respective generative components.

In [ ]:
# ── Inference and Event-Level Clustering Validation ──────────────────
print("Executing Bimodal Branching EM (Stochastic Declustering) on mixture sequence...")
est_mu, est_alpha, est_beta, R_bg, R_trig = bimodal_branching_em(events_mix, T_mix)

# Calculate component responsibilities for each event
# Responsibility = Background Probability + Sum of Offspring Triggering Probabilities
prob_c0 = R_bg[0, :] + R_trig[0, :, :].sum(axis=1)
prob_c1 = R_bg[1, :] + R_trig[1, :, :].sum(axis=1)

pred_labels = (prob_c1 > prob_c0).astype(int)
accuracy = np.mean(pred_labels == labels_true)

# Resolve label-switching symmetry inherent to mixture model EM
if accuracy < 0.5:
    pred_labels = 1 - pred_labels
    accuracy = np.mean(pred_labels == labels_true)
    est_mu, est_alpha, est_beta = est_mu[::-1], est_alpha[::-1], est_beta[::-1]

print("\n── Parameter Recovery & Clustering Report ──────────────")
print(f"Ground Truth Comp 0 : mu=0.30, alpha=0.20, beta=1.00")
print(f"Estimated    Comp 0 : mu={est_mu[0]:.2f}, alpha={est_alpha[0]:.2f}, beta={est_beta[0]:.2f}")
print("-" * 56)
print(f"Ground Truth Comp 1 : mu=1.50, alpha=0.60, beta=2.00")
print(f"Estimated    Comp 1 : mu={est_mu[1]:.2f}, alpha={est_alpha[1]:.2f}, beta={est_beta[1]:.2f}")
print("-" * 56)
print(f"Event Clustering Accuracy : {accuracy * 100:.2f}%")

# ── Visualization: Ground-Truth vs. Predicted Assignments ────────────
plt.figure(figsize=(10, 3))

# Use distinct, academic-friendly hex colors
plt.scatter(events_mix[labels_true==0], np.zeros(sum(labels_true==0)), 
            color='#1f77b4', alpha=0.5, label='Ground Truth: Comp 0')
plt.scatter(events_mix[labels_true==1], np.ones(sum(labels_true==1)), 
            color='#d62728', alpha=0.5, label='Ground Truth: Comp 1')

# Highlight misclassifications to demonstrate declustering performance
errors = events_mix[pred_labels != labels_true]
y_errors = labels_true[pred_labels != labels_true]
if len(errors) > 0:
    plt.scatter(errors, y_errors, color='black', marker='x', s=100, label='Misclassified')

plt.title(rf"Event Clustering via Bimodal Declustering (Accuracy: {accuracy*100:.1f}%)", 
          fontsize=12, fontweight='bold')
plt.yticks([0, 1], ['Component 0', 'Component 1'])
plt.xlabel("Time ($t$)")
plt.legend(loc='center right')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 6. Asymptotic Consistency

We evaluate the Maximum Likelihood Estimator's convergence properties by varying the observation window $T$, generating datasets of increasing sizes.

In [ ]:
# ── Asymptotic Consistency of the MLE ────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from src.simulator import simulate_hawkes
from src.inference import branching_em

true_mu, true_alpha, true_beta = 1.0, 0.6, 1.5
T_list  = [50, 100, 200, 400, 700]
N_REPS  = 20 

mean_errs = {p: [] for p in ['mu', 'alpha', 'beta']}
std_errs  = {p: [] for p in ['mu', 'alpha', 'beta']}
n_means   = []

# Print tabular results
print(f"{'T':>6}  {'N (mean)':>9}  {'mu err':>16}  {'alpha err':>16}  {'beta err':>16}")
print("-" * 70)

for T in T_list:
    runs = {'mu': [], 'alpha': [], 'beta': []}
    ns   = []
    for seed in range(N_REPS):
        rng_i = np.random.default_rng(seed)
        ev    = simulate_hawkes(true_mu, true_alpha, true_beta, T, rng_i)
        ns.append(len(ev))
        try:
            m, a, b, _, _ = branching_em(ev, T)
            runs['mu'].append(abs(m - true_mu))
            runs['alpha'].append(abs(a - true_alpha))
            runs['beta'].append(abs(b - true_beta))
        except Exception:
            continue

    n_mean = int(np.mean(ns))
    n_means.append(n_mean)
    
    for p in ['mu', 'alpha', 'beta']:
        vals = np.array(runs[p])
        if len(vals) > 0:
            mean_errs[p].append(vals.mean())
            std_errs[p].append(vals.std())
        else:
            mean_errs[p].append(np.nan)
            std_errs[p].append(np.nan)

    print(f"{T:>6}  {n_mean:>9}  "
          f"{mean_errs['mu'][-1]:>8.4f} ± {std_errs['mu'][-1]:.4f}  "
          f"{mean_errs['alpha'][-1]:>8.4f} ± {std_errs['alpha'][-1]:.4f}  "
          f"{mean_errs['beta'][-1]:>8.4f} ± {std_errs['beta'][-1]:.4f}")

# ── Visualization: Convergence Rate ──
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

param_cfg = [
    ('mu',    r'$\mu$',     '#2C6FAC'),
    ('alpha', r'$\alpha$',  '#E05A2B'),
    ('beta',  r'$\beta$',   '#3BA35A'),
]

# Panel 1: Mean absolute error with standard deviation bands
ax = axes[0]
for p, label, color in param_cfg:
    means = np.array(mean_errs[p])
    stds  = np.array(std_errs[p])
    valid = ~np.isnan(means)
    
    ax.plot(np.array(n_means)[valid], means[valid], marker='o', color=color,
            linewidth=1.8, label=label)
    ax.fill_between(np.array(n_means)[valid], 
                    means[valid] - stds[valid], 
                    means[valid] + stds[valid],
                    color=color, alpha=0.15)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Expected Number of Events $N$ (log scale)', fontsize=11)
ax.set_ylabel('Mean Absolute Error (log scale)', fontsize=11)
ax.set_title('Estimation Error vs. Sample Size', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)

# Panel 2: Log-log slope fitting against theoretical -0.5 rate
ax = axes[1]
for p, label, color in param_cfg:
    means = np.array(mean_errs[p])
    valid = ~np.isnan(means)
    
    n_valid = np.array(n_means)[valid]
    m_valid = means[valid]
    
    if len(n_valid) > 1:
        log_n = np.log(n_valid)
        log_e = np.log(np.maximum(m_valid, 1e-8))
        slope, intercept = np.polyfit(log_n, log_e, 1)
        fit_line = np.exp(intercept) * n_valid ** slope
        
        ax.plot(n_valid, m_valid, marker='o', color=color,
                linewidth=1.8, label=f'{label} (slope = {slope:.2f})')
        ax.plot(n_valid, fit_line, color=color, linestyle='--',
                linewidth=1.0, alpha=0.7)

# Theoretical Reference Line: sqrt(N) consistency
if len(n_means) > 0 and not np.isnan(mean_errs['mu'][0]):
    ref_intercept = np.exp(np.mean([np.log(mean_errs['mu'][0]),
                                    np.log(mean_errs['alpha'][0]),
                                    np.log(mean_errs['beta'][0])]))
    ax.plot(n_means, ref_intercept * (np.array(n_means) / n_means[0]) ** (-0.5),
            'k:', linewidth=1.2, label=r'Theoretical Rate $\mathcal{O}(N^{-1/2})$')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Expected Number of Events $N$ (log scale)', fontsize=11)
ax.set_ylabel('Mean Absolute Error (log scale)', fontsize=11)
ax.set_title('Empirical Convergence Rate Verification', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Computational Complexity & Optimization

The Expectation-Maximization (EM) algorithm for Hawkes processes faces a computational challenge as the number of events $N$ increases:

1. **Space Complexity**: The full Branching Matrix $P$ is $O(N^2)$. For the UNSW-NB15 dataset (millions of rows), storing this matrix in memory is infeasible. 
2. **Time Complexity**: The E-step involves computing pairwise distances $\Delta t_{ij}$, which is $O(N^2)$.

**Optimization implemented**:
In our code, we utilize the **Markovian property** of the exponential kernel. The recursive term $A_i = \sum_{j < i} e^{-\beta(t_i - t_j)}$ can be computed in **$O(N)$** time using the recurrence:
$$A_i = e^{-\beta(t_i - t_{i-1})}(1 + A_{i-1})$$
This reduction from $O(N^2)$ to $O(N)$ allows our inference engine to handle real-world high-frequency network logs efficiently.

In [ ]:
# ── Section 7: Computational Complexity Benchmark ────────────────────
import time
import numpy as np
import matplotlib.pyplot as plt
from src.simulator import simulate_hawkes

# ── Two complete, numerically equivalent implementations ──────────────

def log_likelihood_naive(events, mu, alpha, beta, T):
    """
    O(N²) log-likelihood via explicit double loop.
    For each event t_i, iterates over all prior events to sum
    the excitation contributions.  Reference implementation only —
    too slow for production use.
    """
    n  = len(events)
    ll = 0.0
    for i in range(n):
        past      = events[:i]
        intensity = mu + alpha * np.sum(np.exp(-beta * (events[i] - past)))
        ll       += np.log(max(intensity, 1e-300))
    comp = mu * T + (alpha / beta) * np.sum(1 - np.exp(-beta * (T - events)))
    return ll - comp


def log_likelihood_fast(events, mu, alpha, beta, T):
    """
    O(N) log-likelihood via the Markovian recursion:

        A_i = exp(-β (t_i − t_{i-1})) · (1 + A_{i-1})

    This recurrence follows from the semigroup property of the
    exponential kernel and reduces the O(N²) double sum to a
    single forward pass, without changing the numerical result.
    """
    n   = len(events)
    A   = np.zeros(n)
    for i in range(1, n):
        A[i] = np.exp(-beta * (events[i] - events[i-1])) * (1 + A[i-1])
    lam  = np.maximum(mu + alpha * A, 1e-300)
    comp = mu * T + (alpha / beta) * np.sum(1 - np.exp(-beta * (T - events)))
    return np.sum(np.log(lam)) - comp


# ── Verify numerical equivalence before timing ────────────────────────
_ev_check = simulate_hawkes(1.0, 0.6, 1.5, 50, np.random.default_rng(9))
_diff = abs(log_likelihood_naive(_ev_check, 1.0, 0.6, 1.5, 50)
            - log_likelihood_fast(_ev_check, 1.0, 0.6, 1.5, 50))
print(f"Numerical difference between implementations: {_diff:.2e}  "
      f"({'✓ negligible' if _diff < 1e-8 else '✗ check code'})")

# ── Timing benchmark ──────────────────────────────────────────────────
T_bench    = [30, 60, 100, 150, 200, 260]
ns         = []
t_naive_ms = []
t_fast_ms  = []
REPS       = 8

rng_bench = np.random.default_rng(0)

for T_b in T_bench:
    ev = simulate_hawkes(1.0, 0.6, 1.5, T_b, rng_bench)
    n  = len(ev)
    ns.append(n)

    t0 = time.perf_counter()
    for _ in range(REPS):
        log_likelihood_naive(ev, 1.0, 0.6, 1.5, T_b)
    t_naive_ms.append((time.perf_counter() - t0) / REPS * 1000)

    t0 = time.perf_counter()
    for _ in range(REPS):
        log_likelihood_fast(ev, 1.0, 0.6, 1.5, T_b)
    t_fast_ms.append((time.perf_counter() - t0) / REPS * 1000)

    speedup = t_naive_ms[-1] / t_fast_ms[-1]
    print(f"N = {n:4d}  |  naive = {t_naive_ms[-1]:7.2f} ms  "
          f"|  fast = {t_fast_ms[-1]:6.3f} ms  |  speedup = {speedup:.0f}×")

# ── Fit log-log slopes to confirm complexity class ────────────────────
ns_arr = np.array(ns)
slope_naive, ic_naive = np.polyfit(np.log(ns_arr), np.log(t_naive_ms), 1)
slope_fast,  ic_fast  = np.polyfit(np.log(ns_arr), np.log(t_fast_ms),  1)

fit_naive = np.exp(ic_naive) * ns_arr ** slope_naive
fit_fast  = np.exp(ic_fast)  * ns_arr ** slope_fast

# ── Plot ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(ns, t_naive_ms, 'o-', color='#E05A2B', linewidth=1.8,
        label=f'Naive O(N²)  [fitted slope = {slope_naive:.2f}]')
ax.plot(ns, t_fast_ms,  's-', color='#2C6FAC', linewidth=1.8,
        label=f'Fast  O(N)   [fitted slope = {slope_fast:.2f}]')
ax.set_xlabel('Number of Events  N',   fontsize=11)
ax.set_ylabel('Wall-clock time  (ms)', fontsize=11)
ax.set_title('Runtime vs Dataset Size', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.loglog(ns, t_naive_ms, 'o',  color='#E05A2B', markersize=6)
ax.loglog(ns, t_fast_ms,  's',  color='#2C6FAC', markersize=6)
ax.loglog(ns, fit_naive,  '--', color='#E05A2B', linewidth=1.5,
          label=f'Naive  slope = {slope_naive:.2f}  (theory: 2.0)')
ax.loglog(ns, fit_fast,   '--', color='#2C6FAC', linewidth=1.5,
          label=f'Fast   slope = {slope_fast:.2f}  (theory: 1.0)')
ax.set_xlabel('Number of Events  N  (log scale)', fontsize=11)
ax.set_ylabel('Wall-clock time  (log scale)',      fontsize=11)
ax.set_title('Log-log Plot: Confirming Complexity Class', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)

plt.suptitle('Computational Complexity Benchmark: O(N²) vs O(N)', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Discussion & Conclusion

This project reveals critical insights into the inference of Hawkes processes and their mixture formulations:

1. **The Inevitability of Component Collapse in Naive Mixtures**:
   Our experiments empirically demonstrate that directly applying Gaussian-Mixture-style likelihoods $p(t_i) = \pi_1 \lambda_1(t_i) + \pi_2 \lambda_2(t_i)$ to sequential point processes fails entirely. As demonstrated in Section 4, the mixture weight $\pi$ rapidly degenerates to an extreme absorption state (e.g., $\pi \to \{1, 0\}$) within just 10-15 iterations. Mathematically, this is caused by the **shared history filtration** $\mathcal{H}_{t}$. Both components rely on the exact same recursive accumulation term $A_i = \sum e^{-\beta \Delta t}$, leading to severe multicollinearity on the non-convex likelihood surface. The optimizer inevitably kills one component to reduce model complexity.

2. **Declustering as the Theoretical Resolution**:
   By shifting to a Bimodal Branching EM (Section 5), we correctly aligned our inference with the true generative mechanism. Instead of assigning a global weight $\pi$ to the entire timeline, the algorithm computes the probability of each *individual event* being a background arrival ($p_{ii}$) versus a triggered offspring ($p_{ij}$). This branching mechanism effectively breaks the collinearity, recovering the ground-truth parameter sets $(\mu, \alpha, \beta)$ and achieving robust event-level clustering accuracy.

3. **Convergence and Asymptotic Properties**:
   The theoretical guarantees of our implementation were rigorously validated in two dimensions. First, the algorithm exhibits strict monotonic increase in log-likelihood and **linear convergence** in the parameter sequence, evidenced by the logarithmic decay of the distance $|\theta^{(n+1)} - \theta^{(n)}|$ (Section 3). Second, the Maximum Likelihood Estimator maintains **asymptotic consistency**, with absolute estimation errors strictly decaying as the dataset size $N \to \infty$ (Section 6).

In conclusion, extending inference from single to mixture Hawkes processes requires a fundamental shift in event attribution (declustering). This resolution bridges the gap between simulated cascade generation and reliable parameter recovery.

## References

1. **Hawkes, A. G. (1971).** "Spectra of some self-exciting and mutually exciting point processes." *Biometrika*, 58(1), 83–90.  
   *(The foundational paper for the self-exciting model used in Section 1.)*

2. **Dempster, A. P., Laird, N. M., & Rubin, D. B. (1977).** "Maximum Likelihood from Incomplete Data via the EM Algorithm." *Journal of the Royal Statistical Society: Series B*, 39(1), 1–38.  
   *(The theoretical basis for the monotonic convergence of the EM algorithm shown in Section 3.)*

3. **Ogata, Y. (1981).** "On Lewis' simulation method for point processes." *IEEE Transactions on Information Theory*.  
   *(Standard citation for the thinning algorithm used in your `simulator.py`.)*

4. **Zhuang, J., Ogata, Y., & Vere-Jones, D. (2002).** "Stochastic declustering of space-time earthquake occurrences." *Journal of the American Statistical Association*, 97(458), 369–380.  
   *(CRITICAL: Your `branching_em` implementation is technically a "Stochastic Declustering" method, first formalized here.)*